### 1. VENDAS

-   QUANTIDADE DE VENDAS POR PRODUTO E CATEGORIA
-   QUANTIDADE DE VENDAS POR REGIAO E ESTADO
-   RECEITA TOTAL MENSAL, ANUAL E DIÁRIA
-   TICKET MEDIO
-   MARGEM DE LUCRO
-   DATA DA VENDA


In [0]:
import pandas as pd

# Lê os arquivos
df1 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Vendas.csv')
df2 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Clientes.csv')
df3 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Lojas.csv')
df4 = pd.read_csv('/Volumes/databricks_database/volumes/arquivos/Produtos.csv')

In [0]:
df1 = df1.rename(columns={
    'Venda_ID': 'id_venda',
    'Loja_ID': 'id_loja',
    'Produto_ID': 'id_produto',
    'Cliente_ID': 'id_cliente',
    'Colaborador_ID': 'id_colaborador',
    'Quantidade': 'quantidade',
    'Preço Unitário': 'preco_unitario',
    'Data da Venda': 'data_venda',
    'Canal de Venda': 'canal_venda'
})

df1['quantidade'] = pd.to_numeric(df1['quantidade'], errors='coerce')
df1['preco_unitario'] = pd.to_numeric(df1['preco_unitario'], errors='coerce')
df1 = df1.dropna(subset=['quantidade', 'preco_unitario'])
df1['data_venda'] = pd.to_datetime(df1['data_venda'], errors='coerce')
df_vendas = df1


In [0]:
spark_df_vendas = spark.createDataFrame(df_vendas)
spark_df_vendas.createOrReplaceTempView("temp_vendas")

In [0]:
%sql
SELECT *
FROM temp_vendas

In [0]:

df2 = df2.rename(columns={
    'Cliente_ID': 'id_cliente',
    'Nome': 'nome_cliente',
    'Idade': 'idade_cliente',
    'Género': 'genero',
    'Cidade': 'cidade_cliente'
})
df_clientes = df2[['id_cliente', 'nome_cliente', 'idade_cliente', 'genero', 'cidade_cliente']]

In [0]:
spark_df_clientes = spark.createDataFrame(df_clientes)
spark_df_clientes.createOrReplaceTempView("temp_clientes")

In [0]:
%sql
SELECT *
FROM temp_clientes

In [0]:
df3 = df3.rename(columns={
    'Loja_ID': 'id_loja',
    'Nome': 'nome_loja',
    'Região': 'regiao_loja',
    'Cidade': 'cidade_loja',
    'Tipo': 'tipo_loja'
})
df_lojas = df3[['id_loja', 'nome_loja', 'regiao_loja', 'cidade_loja', 'tipo_loja']]

In [0]:
spark_df_lojas = spark.createDataFrame(df_lojas)
spark_df_lojas.createOrReplaceTempView("temp_lojas")

In [0]:
%sql
SELECT *
FROM temp_lojas

In [0]:
df4 = df4.rename(columns={
    'Produto_ID': 'id_produto',
    'Nome': 'nome_produto',
    'Categoria': 'categoria',
    'Cor': 'cor',
    'Descrição': 'descricao',
    'Tamanho': 'tamanho',
    'Preço': 'preco',
    'Custo_Aquisição': 'custo_aquisicao',
    'Imagem': 'imagem'
})
df_produtos= df4[['id_produto', 'nome_produto', 'categoria']]
df_produtos.head()

In [0]:
import pandas as pd

# converter para pandas
df_produtos= df4[['id_produto', 'nome_produto', 'categoria']]

roupas = [
"Camiseta Nike","Camiseta Adidas","Camiseta Puma","Camisa Polo","Camisa Social",
"Jaqueta Jeans","Jaqueta Corta Vento","Moletom Nike","Moletom Adidas",
"Calça Jeans","Calça Moletom","Calça Cargo","Bermuda Jeans","Bermuda Esportiva",
"Vestido Casual","Vestido Longo","Blusa Feminina","Blusa Cropped",
"Saia Jeans","Saia Midi","Regata Esportiva","Regata Nike","Regata Adidas"
]

calcados = [
"Tenis Nike Air","Tenis Adidas Ultraboost","Tenis Puma Runner","Tenis Vans Old Skool",
"Tenis Converse All Star","Bota Timberland","Bota Coturno","Sapato Social",
"Sapato Oxford","Sandalia Feminina","Sandalia Rasteira","Chinelo Havaianas",
"Chinelo Slide Nike","Tenis Nike SB","Tenis Adidas Runfalcon","Tenis Mizuno Wave",
"Tenis Asics Gel","Tenis New Balance","Tenis Olympikus","Tenis Fila"
]

i_roupa = 0
i_calcado = 0

for i,row in df_produtos.iterrows():
    
    if row["categoria"] == "Roupas":
        df_produtos.at[i,"nome_produto"] = roupas[i_roupa % len(roupas)]
        i_roupa += 1
        
    else:
        df_produtos.at[i,"nome_produto"] = calcados[i_calcado % len(calcados)]
        i_calcado += 1

In [0]:
spark_df_produtos = spark.createDataFrame(df_produtos)
spark_df_produtos.createOrReplaceTempView("temp_produtos")

In [0]:
%sql
SELECT *
FROM temp_produtos

In [0]:
%sql
SELECT *
FROM temp_clientes
group by all

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW b1 AS
      SELECT v.id_venda
            , YEAR(v.data_venda) AS ano_venda
            , CAST(v.data_venda AS DATE) AS data_formatada
            , v.data_venda
            , v.canal_venda

            --infos loja
            , v.id_loja
            , l.nome_loja
            , l.regiao_loja
            , l.cidade_loja

            -- infos produto
            , v.id_produto
            , p.categoria
            , p.nome_produto

            -- infos clientes
            , v.id_cliente
            , c.nome_cliente
            , c.idade_cliente
            , c.genero
            , c.cidade_cliente

            -- infos vendas
            , v.id_colaborador
            , v.quantidade
            , v.preco_unitario
            , v.quantidade * v.preco_unitario AS receita_total

      FROM temp_vendas AS v 

      LEFT JOIN temp_clientes as c
      ON v.id_cliente = c.id_cliente

      LEFT JOIN temp_lojas as l 
      ON v.id_loja = l.id_loja

      LEFT JOIN temp_produtos as p 
      ON v.id_produto = p.id_produto

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rank_produto_ano AS
WITH ano_produto AS (
  SELECT
    ano_venda,
    --id_produto,
    nome_produto,
    categoria,
    SUM(receita_total) AS receita_ano,
    SUM(quantidade)    AS qtd_ano
  FROM b1
  GROUP BY 1,2,3
),
ranked AS (
  SELECT
    *,
    dense_rank() OVER (
      PARTITION BY ano_venda
      ORDER BY receita_ano DESC
    ) AS rank_receita
  FROM ano_produto
)
SELECT
  *,
  CASE WHEN rank_receita <= 10 THEN 1 ELSE 0 END AS fl_dentro_top10
FROM ranked;

In [0]:
%sql
CREATE OR REPLACE TABLE databricks_database.gold.visao_vendas_dashboard AS
  SELECT
    b1.*,
    r.rank_receita,
    r.fl_dentro_top10
  FROM b1
  LEFT JOIN rank_produto_ano r
    ON b1.ano_venda = r.ano_venda
  AND b1.nome_produto = r.nome_produto;

In [0]:
%sql
select * from databricks_database.gold.visao_vendas_dashboard

In [0]:
%sql
select * 
from databricks_database.gold.visao_vendas_dashboard
where data_formatada = '2024-01-01'